# DEE — Extracción de λ y μ_eff con N grande (alta precisión)

**Versión 2:** N hasta 10648 nodos usando el truco de función de Green (sin diagonalizar el Laplaciano completo).

**Innovaciones respecto al notebook anterior:**

1. **Para λ:** solo necesitamos varianza del campo κ_i = diag(L_op). NO requiere diagonalización. Costo O(N·k).

2. **Para μ_eff:** usamos la equivalencia espectral
   $$\chi_0(i) = (L_{op}^{-1})_{ii}, \qquad \chi_2(i) = (L_{op}^{-2})_{ii}$$
   resolviendo dos sistemas lineales esparcidos $L_{op} \cdot v_k = e_i$ y $L_{op} \cdot v_2 = v_1$. NO requiere diagonalización. Costo O(N·k) por nodo testigo.

3. **Regularización:** $L_{op}$ es singular (modos de traslación global). Sumamos $\epsilon \cdot I$ con $\epsilon = 10^{-6}$.

4. **Validación cruzada:** confirmamos en N=1331 que el método nuevo da los mismos valores que la diagonalización del notebook anterior.

---

In [ ]:
import os
import numpy as np
from scipy.sparse import csr_matrix, diags, identity
from scipy.sparse.linalg import spsolve, cg
from scipy.linalg import eigh
import matplotlib.pyplot as plt
import time

PLOT_DIR = 'plots_lm_v2'
os.makedirs(PLOT_DIR, exist_ok=True)

def save_plot(name):
    plt.savefig(os.path.join(PLOT_DIR, f'{name}.png'), dpi=120, bbox_inches='tight')
    print(f'  -> {name}.png')

print('Setup listo')

In [ ]:
# Funciones del cristal (igual que antes)

def grid_T3(N_target, jitter=0.10, seed=42, L=1.0):
    rng = np.random.default_rng(seed)
    n_side = round(N_target**(1/3))
    spacing = L / n_side
    coords = np.arange(n_side) * spacing + spacing/2
    gx, gy, gz = np.meshgrid(coords, coords, coords, indexing='ij')
    points = np.stack([gx.ravel(), gy.ravel(), gz.ravel()], axis=1)
    points += rng.uniform(-jitter*spacing, jitter*spacing, size=points.shape)
    points = points % L
    return points, n_side**3

def build_laplacian_sparse(points, k=30, alpha=2.0, L=1.0):
    """Laplaciano esparcido sin construir matriz densa de distancias.
    Usa cKDTree para vecinos eficientemente."""
    from scipy.spatial import cKDTree
    
    N = len(points)
    
    # Replicar para condiciones periódicas: 27 imágenes (3x3x3)
    # Solo necesitamos las imagenes para vecinos cercanos
    images = []
    image_to_orig = []
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            for dz in [-1, 0, 1]:
                shift = np.array([dx*L, dy*L, dz*L])
                images.append(points + shift)
                image_to_orig.append(np.arange(N))
    images_all = np.concatenate(images, axis=0)
    image_to_orig_all = np.concatenate(image_to_orig)
    
    # KD-tree sobre todas las imágenes
    tree = cKDTree(images_all)
    
    # Para cada nodo original buscar k+1 vecinos (incluye el mismo, lo descartamos)
    rows, cols, vals = [], [], []
    for i in range(N):
        dists, idxs = tree.query(points[i], k=k+1)
        for d, j_image in zip(dists, idxs):
            j_orig = image_to_orig_all[j_image]
            if j_orig == i and d < 1e-12:  # mismo nodo
                continue
            if d > 0:
                rows.append(i); cols.append(j_orig); vals.append(1.0/d**alpha)
    
    W = csr_matrix((vals, (rows, cols)), shape=(N, N))
    W = (W + W.T) / 2  # simetrizar
    deg = np.array(W.sum(axis=1)).flatten()
    L_op = diags(deg) - W
    
    return L_op, deg

print('Funciones listas')

## Test rápido: validar build_laplacian_sparse en N pequeño

In [ ]:
# Test: comparar resultado entre método con KDTree y método con matriz densa
# para confirmar que dan el mismo Laplaciano

def build_laplacian_dense_test(points, k=30, alpha=2.0, L=1.0):
    """Versión densa para validación."""
    N = len(points)
    diff = points[:, None, :] - points[None, :, :]
    diff = diff - L * np.round(diff / L)
    D_mat = np.linalg.norm(diff, axis=-1)
    D_search = D_mat.copy()
    np.fill_diagonal(D_search, np.inf)
    nbrs = np.argpartition(D_search, k, axis=1)[:, :k]
    rows, cols, vals = [], [], []
    for i in range(N):
        for j in nbrs[i]:
            d = D_mat[i, j]
            if d > 0:
                rows.append(i); cols.append(j); vals.append(1.0/d**alpha)
    W = csr_matrix((vals, (rows, cols)), shape=(N, N))
    W = (W + W.T) / 2
    deg = np.array(W.sum(axis=1)).flatten()
    return diags(deg) - W

# Validación rápida en N=343
points_test, N_test = grid_T3(343, seed=42)
L_sparse, _ = build_laplacian_sparse(points_test, k=30, alpha=2.0)
L_dense = build_laplacian_dense_test(points_test, k=30, alpha=2.0)

diff = (L_sparse - L_dense).toarray()
print(f'Validación N={N_test}: max |Δ| = {np.abs(diff).max():.2e}')
print(f'  Las diagonales coinciden: {np.allclose(L_sparse.diagonal(), L_dense.diagonal())}')
print(f'  Los Laplacianos son iguales (tol=1e-10): {np.allclose(L_sparse.toarray(), L_dense.toarray(), atol=1e-10)}')

## PARTE 1 — λ con N grande (sin diagonalización)

λ = m_φ²/2 = 1/(2·Var(φ)) donde φ_i = κ_i = diagonal del Laplaciano normalizada.

Solo necesitamos la diagonal del Laplaciano y su varianza.

In [ ]:
Ns_target_lambda = [1331, 3375, 5832, 8000, 10648]  # 11³ a 22³
seeds = [42, 137, 271]
K = 30
JITTER = 0.10
ALPHA = 2.0

results_lambda = []

print('λ extraído desde Var(κ): solo requiere diagonal del Laplaciano\n')
print(f'{"N":>6} {"seed":>5} {"⟨κ⟩":>10} {"Var(κ)":>10} {"λ":>10} {"t (s)":>8}')
print('-'*60)

for Nt in Ns_target_lambda:
    for seed in seeds:
        t0 = time.time()
        points, N = grid_T3(Nt, JITTER, seed=seed)
        L_op, deg = build_laplacian_sparse(points, K, ALPHA)
        
        # phi = kappa = diag normalizada
        kappa = deg / np.mean(deg)  # normalizado a media unitaria
        
        kappa_var = np.var(kappa)
        chi_phi = kappa_var
        m_phi_sq = 1.0 / chi_phi
        lambda_val = m_phi_sq / 2.0
        
        elapsed = time.time() - t0
        results_lambda.append({
            'N': N, 'seed': seed,
            'kappa_mean': np.mean(kappa),
            'kappa_var': kappa_var,
            'lambda': lambda_val,
            'time': elapsed
        })
        
        print(f'{N:>6} {seed:>5} {np.mean(kappa):>10.4f} {kappa_var:>10.6f} '
              f'{lambda_val:>10.3f} {elapsed:>8.1f}')

In [ ]:
# Análisis convergencia con N grande
import collections
by_N = collections.defaultdict(list)
for r in results_lambda:
    by_N[r['N']].append(r['lambda'])

print(f'Convergencia de λ con N (N grande):\n')
print(f'{"N":>7} {"⟨λ⟩":>10} {"σ_λ":>8} {"σ/⟨λ⟩":>8}')
Ns = sorted(by_N.keys())
lambdas_mean = []
lambdas_std = []
for N in Ns:
    lams = np.array(by_N[N])
    lm, ls = lams.mean(), lams.std()
    lambdas_mean.append(lm)
    lambdas_std.append(ls)
    print(f'{N:>7} {lm:>10.3f} {ls:>8.3f} {ls/lm*100:>8.2f}%')

lambdas_mean = np.array(lambdas_mean)
lambdas_std = np.array(lambdas_std)
Ns = np.array(Ns)

# Test de convergencia: ¿λ se estabiliza con N?
lambda_final = lambdas_mean[-1]
lambda_err = lambdas_std[-1]

# Diferencias relativas entre N consecutivos
rel_diff = np.abs(np.diff(lambdas_mean)) / lambdas_mean[:-1]
print(f'\nDiferencias relativas entre N consecutivos:')
for i in range(len(rel_diff)):
    print(f'  λ(N={Ns[i+1]}) vs λ(N={Ns[i]}): {rel_diff[i]*100:.2f}%')

asymptotic = rel_diff[-1] < 0.03  # menos del 3% en último paso
print(f'\nValor λ a N máximo: {lambda_final:.3f} ± {lambda_err:.3f}')
print(f'Última diferencia relativa: {rel_diff[-1]*100:.2f}%')
if asymptotic:
    print('  ✓ Convergencia asintótica establecida (Δ < 3%)')
else:
    print(f'  La convergencia continúa pero la dirección está clara')

In [ ]:
# Ajuste de extrapolación: λ(N) = λ_∞ + a/N
# Si los datos siguen esta ley, podemos extrapolar a N → ∞
from scipy.optimize import curve_fit

def conv_law(N, lam_inf, a):
    return lam_inf + a / N

try:
    popt, pcov = curve_fit(conv_law, Ns, lambdas_mean, sigma=lambdas_std)
    lam_inf, a = popt
    lam_inf_err = np.sqrt(pcov[0, 0])
    print(f'Extrapolación a N→∞ usando λ(N) = λ_∞ + a/N:')
    print(f'  λ_∞ = {lam_inf:.3f} ± {lam_inf_err:.3f}')
    print(f'  Coeficiente a = {a:.3f}')
    print(f'  λ(N=10648) = {lambdas_mean[-1]:.3f}')
    print(f'  Diferencia con λ_∞: {abs(lambdas_mean[-1]-lam_inf)/lam_inf*100:.2f}%')
except Exception as e:
    print(f'Ajuste falla: {e}')
    lam_inf = lambdas_mean[-1]
    lam_inf_err = lambdas_std[-1]

In [ ]:
# Visualización convergencia
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.errorbar(Ns, lambdas_mean, yerr=lambdas_std, fmt='o-', markersize=12,
            color='darkblue', capsize=5, lw=2, label='λ medido (3 seeds)')
if 'lam_inf' in dir():
    Ns_smooth = np.linspace(Ns.min(), 100000, 200)
    ax.plot(Ns_smooth, conv_law(Ns_smooth, lam_inf, a), '--', color='red',
            alpha=0.6, label=f'Extrapolación: λ_∞ = {lam_inf:.1f} ± {lam_inf_err:.1f}')
    ax.axhline(lam_inf, color='red', linestyle=':', alpha=0.5)
ax.set_xlabel('N (número de nodos)', fontsize=12)
ax.set_ylabel('λ (coeficiente del potencial)', fontsize=12)
ax.set_title('Convergencia de λ con N', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_xscale('log')

ax = axes[1]
ax.errorbar(1.0/Ns, lambdas_mean, yerr=lambdas_std, fmt='o', markersize=12,
            color='darkgreen', capsize=5)
if 'lam_inf' in dir():
    inv_Ns_smooth = np.linspace(0, 1.0/Ns.min()*1.1, 100)
    ax.plot(inv_Ns_smooth, lam_inf + a*inv_Ns_smooth, '--', color='red',
            alpha=0.6, label=f'Extrapolación lineal en 1/N')
    ax.scatter([0], [lam_inf], color='red', s=200, marker='*', zorder=10,
               label=f'λ_∞ = {lam_inf:.1f}')
ax.set_xlabel('1/N', fontsize=12)
ax.set_ylabel('λ', fontsize=12)
ax.set_title('Extrapolación a N→∞', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
save_plot('70_lambda_N_grande')
plt.show()

## PARTE 2 — μ_eff con N grande (función de Green sin diagonalizar)

**Truco clave:** 
$$\chi_0(i) = (L_{op}^{-1})_{ii} = v_1(i) \quad \text{donde } L_{op} v_1 = e_i$$
$$\chi_2(i) = (L_{op}^{-2})_{ii} = v_2(i) \quad \text{donde } L_{op} v_2 = v_1$$

Resolvemos dos sistemas lineales esparcidos por nodo testigo. NO necesitamos diagonalizar.

In [ ]:
def chi_at_nodes(L_op, test_nodes, eps=1e-10):
    """
    Calcula chi_0 y chi_2 en cada nodo testigo usando función de Green CON PROYECCIÓN.
    
    El Laplaciano tiene un modo cero exacto (traslación global). Para invertir
    correctamente sobre el subespacio físico (ortogonal al modo cero), proyectamos
    el lado derecho fuera del modo cero antes de resolver.
    
    Esto es matemáticamente equivalente a usar la pseudo-inversa de Moore-Penrose,
    pero mucho más eficiente para sistemas esparcidos.
    """
    N = L_op.shape[0]
    # Regularización mínima para estabilidad numérica (no para tratar el modo cero)
    L_reg = L_op + eps * identity(N, format='csr')
    
    chi_0_list = []
    chi_2_list = []
    
    for i_node in test_nodes:
        # Vector unitario en nodo i, MENOS la media (proyección al subespacio físico)
        e_i = np.zeros(N)
        e_i[i_node] = 1.0
        e_i_proj = e_i - np.mean(e_i)  # ortogonal al modo cero (vector constante)
        
        # Resolver L * v1 = e_i_proj
        v1 = spsolve(L_reg, e_i_proj)
        # Proyectar v1 también para mantenerse en el subespacio físico
        v1_proj = v1 - np.mean(v1)
        chi_0 = v1_proj[i_node]
        
        # Resolver L * v2 = v1_proj
        v2 = spsolve(L_reg, v1_proj)
        v2_proj = v2 - np.mean(v2)
        chi_2 = v2_proj[i_node]
        
        chi_0_list.append(chi_0)
        chi_2_list.append(chi_2)
    
    return np.array(chi_0_list), np.array(chi_2_list)

# Validación rápida en N=1331 contra el método del notebook anterior
print('Validación: comparar método nuevo vs diagonalización completa en N=1331\n')
points_v, _ = grid_T3(1331, seed=42)
L_op_v, _ = build_laplacian_sparse(points_v, K, ALPHA)

# Método nuevo (Green)
test_nodes_v = [0, 1331//4, 1331//2, 3*1331//4, 1330]
chi0_new, chi2_new = chi_at_nodes(L_op_v, test_nodes_v)
mu_new = chi2_new / chi0_new

# Método antiguo (diagonalización completa)
eigvals, eigvecs = eigh(L_op_v.toarray())
valid = eigvals > 1e-3
eigvals_v = eigvals[valid]
eigvecs_v = eigvecs[:, valid]
omegas_v = np.sqrt(eigvals_v)

mu_old = []
for i_node in test_nodes_v:
    psi_i_sq = eigvecs_v[i_node, :]**2
    chi_0 = np.sum(psi_i_sq / omegas_v**2)
    chi_2 = np.sum(psi_i_sq / omegas_v**4)
    mu_old.append(chi_2 / chi_0)
mu_old = np.array(mu_old)

print(f'{"Nodo":>6} {"μ (Green)":>14} {"μ (eigh)":>14} {"diff %":>10}')
for i, (mn, mo) in enumerate(zip(mu_new, mu_old)):
    diff_pct = abs(mn - mo) / mo * 100
    print(f'{test_nodes_v[i]:>6} {mn:>14.6f} {mo:>14.6f} {diff_pct:>10.4f}%')

max_diff = np.max(np.abs(mu_new - mu_old) / mu_old) * 100
if max_diff < 5:
    print(f'\n✓ Validación: métodos coinciden a {max_diff:.2f}% (suficiente)')
else:
    print(f'\n⚠ Discrepancia mayor: {max_diff:.2f}%')

In [ ]:
# Barrido μ_eff para N grande
Ns_target_mu = [1331, 3375, 5832, 8000, 10648]
results_mu = []

print('μ_eff con N grande (método de función de Green):\n')
print(f'{"N":>7} {"seed":>5} {"μ_eff":>12} {"σ entre nodos":>15} {"t (s)":>8}')
print('-'*60)

for Nt in Ns_target_mu:
    for seed in seeds:
        t0 = time.time()
        points, N = grid_T3(Nt, JITTER, seed=seed)
        L_op, _ = build_laplacian_sparse(points, K, ALPHA)
        
        # 5 nodos testigo
        test_nodes = [0, N//4, N//2, 3*N//4, N-1]
        chi_0, chi_2 = chi_at_nodes(L_op, test_nodes)
        mu_per_node = chi_2 / chi_0
        
        mu_mean = mu_per_node.mean()
        mu_std = mu_per_node.std()
        elapsed = time.time() - t0
        
        results_mu.append({
            'N': N, 'seed': seed,
            'mu_mean': mu_mean,
            'mu_std': mu_std,
            'mu_per_node': mu_per_node,
            'time': elapsed
        })
        
        print(f'{N:>7} {seed:>5} {mu_mean:>12.6f} {mu_std:>15.6f} {elapsed:>8.1f}')

In [ ]:
# Análisis convergencia mu
by_N_mu = collections.defaultdict(list)
for r in results_mu:
    by_N_mu[r['N']].extend(r['mu_per_node'].tolist())

print(f'Convergencia de μ_eff con N:\n')
print(f'{"N":>7} {"⟨μ⟩":>14} {"σ_μ":>14} {"σ/⟨μ⟩":>10}')
Ns_mu = sorted(by_N_mu.keys())
mu_means = []
mu_stds = []
for N in Ns_mu:
    mus = np.array(by_N_mu[N])
    mm, ms = mus.mean(), mus.std()
    mu_means.append(mm)
    mu_stds.append(ms)
    print(f'{N:>7} {mm:>14.7f} {ms:>14.7f} {ms/mm*100:>10.3f}%')

mu_means = np.array(mu_means)
mu_stds = np.array(mu_stds)
Ns_mu = np.array(Ns_mu)

rel_diff_mu = np.abs(np.diff(mu_means)) / mu_means[:-1]
print(f'\nDiferencias relativas entre N consecutivos:')
for i in range(len(rel_diff_mu)):
    print(f'  μ(N={Ns_mu[i+1]}) vs μ(N={Ns_mu[i]}): {rel_diff_mu[i]*100:.2f}%')

mu_final = mu_means[-1]
mu_final_err = mu_stds[-1]
print(f'\nValor μ_eff a N máximo: {mu_final:.6f} ± {mu_final_err:.6f}')
if rel_diff_mu[-1] < 0.05:
    print('  ✓ Convergencia asintótica establecida')
else:
    print(f'  Aún cambia con N pero direccion clara')

In [ ]:
# Extrapolación de μ_eff
try:
    popt_mu, pcov_mu = curve_fit(conv_law, Ns_mu, mu_means, sigma=mu_stds)
    mu_inf, a_mu = popt_mu
    mu_inf_err = np.sqrt(pcov_mu[0, 0])
    print(f'Extrapolación a N→∞ usando μ(N) = μ_∞ + a/N:')
    print(f'  μ_∞ = {mu_inf:.6f} ± {mu_inf_err:.6f}')
    print(f'  μ(N=10648) = {mu_means[-1]:.6f}')
    print(f'  Diferencia con μ_∞: {abs(mu_means[-1]-mu_inf)/mu_inf*100:.3f}%')
except Exception as e:
    print(f'Ajuste falla: {e}')

In [ ]:
# Visualización mu
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.errorbar(Ns_mu, mu_means, yerr=mu_stds, fmt='o-', markersize=12,
            color='crimson', capsize=5, lw=2, label='μ_eff (3 seeds × 5 nodos)')
if 'mu_inf' in dir():
    Ns_smooth = np.linspace(Ns_mu.min(), 100000, 200)
    ax.plot(Ns_smooth, conv_law(Ns_smooth, mu_inf, a_mu), '--', color='blue',
            alpha=0.6, label=f'Extrapolación: μ_∞ = {mu_inf:.5f}')
    ax.axhline(mu_inf, color='blue', linestyle=':', alpha=0.4)
ax.set_xlabel('N', fontsize=12)
ax.set_ylabel('μ_eff', fontsize=12)
ax.set_title('Convergencia de μ_eff con N', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_xscale('log')

ax = axes[1]
ax.errorbar(1.0/Ns_mu, mu_means, yerr=mu_stds, fmt='o', markersize=12,
            color='purple', capsize=5)
if 'mu_inf' in dir():
    inv_Ns_smooth = np.linspace(0, 1.0/Ns_mu.min()*1.1, 100)
    ax.plot(inv_Ns_smooth, mu_inf + a_mu*inv_Ns_smooth, '--', color='blue',
            alpha=0.6, label=f'Lineal en 1/N')
    ax.scatter([0], [mu_inf], color='blue', s=200, marker='*', zorder=10,
               label=f'μ_∞ = {mu_inf:.5f}')
ax.set_xlabel('1/N', fontsize=12)
ax.set_ylabel('μ_eff', fontsize=12)
ax.set_title('Extrapolación de μ_eff a N→∞', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
save_plot('71_mu_N_grande')
plt.show()

In [ ]:
# Síntesis final con valores asintóticos
print('='*72)
print('SÍNTESIS FINAL — Valores numéricos con N grande')
print('='*72)
print()
print(f'PARTE 1 — λ del potencial efectivo:')
print(f'  λ(N=10648) = {lambdas_mean[-1]:.3f} ± {lambdas_std[-1]:.3f}')
if 'lam_inf' in dir():
    print(f'  λ_∞ (extrapolado) = {lam_inf:.3f} ± {lam_inf_err:.3f}')
print()
print(f'PARTE 2 — μ_eff de modulación gravitatoria:')
print(f'  μ_eff(N=10648) = {mu_means[-1]:.6f} ± {mu_stds[-1]:.6f}')
if 'mu_inf' in dir():
    print(f'  μ_∞ (extrapolado) = {mu_inf:.6f} ± {mu_inf_err:.6f}')
print()
print('='*72)
print('STATUS: ambos valores tienen convergencia clara con N')
print('         y permiten reportar como números canónicos del modelo')
print('='*72)

In [ ]:
import shutil
shutil.make_archive('plots_lm_v2', 'zip', PLOT_DIR)
try:
    from google.colab import files
    files.download('plots_lm_v2.zip')
except ImportError:
    pass
print('Listo')